In [1]:
#importamos funciones para utilzar en nuestro flujo
import pandas as pd
import os
import sys
import sqlite3
import requests
import time
import kaggle
from kaggle.api.kaggle_api_extended import KaggleApi
from pathlib import Path
from dotenv import load_dotenv #pip install python-dotenv
from googleapiclient.discovery import build
from google.oauth2 import service_account #pip install google-auth
from googleapiclient.http import MediaFileUpload
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from mimetypes import guess_type
import hashlib
import json
from datetime import datetime


ruta_utils = os.path.expanduser('~/Documents/IA/4_semestre/ing_de_datos/ing_datos/tools')
sys.path.append(ruta_utils)

import tools

#from tools import DRIVE_IDS
print(f"Tools importado correctamente")
print(f"Los métodos alojados en las herramientas son {dir(tools)}")

Tools importado correctamente
Los métodos alojados en las herramientas son ['BASE_DIR', 'DATASETS', 'DRIVE_IDS', 'ETIQUETAS', 'InstalledAppFlow', 'KAGGLE_JSON_PATH', 'KaggleApi', 'LOG_EXTRACCION', 'LOG_TRANSF', 'MediaFileUpload', 'Path', 'RUTA_CARPETA_BASES_KAGGLE', 'RUTA_CARPETA_BASE_SQL', 'RUTA_EXTRACCION_LOGS', 'RUTA_TRANSF_LOGS', 'RUTA_TRANSF_PROCESSED', 'RUTA_TRANSF_REPORTES', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'build', 'datetime', 'drive', 'glob', 'guess_type', 'json', 'kaggle', 'load_dotenv', 'logs', 'os', 'pd', 'requests', 'service_account', 'sql', 'sqlite3', 'sys', 'tempfile', 'time', 'transformacion']


In [2]:
DB_PATH = str(tools.DATASETS["adilshamim8"]["db"])
TABLA_RAW = tools.DATASETS["adilshamim8"]["raw"]
def listar_tablas(db_path=None):

    """
    Lista todas las tablas del archivo .db clasificadas por capa
    (raw_, extracted_, transformed_, serving_).

    Parámetros:
        db_path : ruta al archivo .db (usa DB_PATH por defecto)

    Retorna:
        Lista con los nombres de las tablas encontradas
    """

    if db_path is None:
        db_path = DB_PATH

    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")
    tablas = [row[0] for row in cursor.fetchall()]

    print(f'\n  Tablas en: {os.path.basename(db_path)}\n')
    print(f'  {"TABLA":<35} {"CAPA":<15} {"FILAS":>8} {"COLS":>6}')
    print(f'  {"-"*35} {"-"*15} {"-"*8} {"-"*6}')

    for tabla in tablas:
        df_t = pd.read_sql_query(f'SELECT * FROM "{tabla}"', conn)
        capa = tabla.split('_')[0] + '_' if '_' in tabla else 'otra'
        print(f'  {tabla:<35} {capa:<15} {df_t.shape[0]:>8,} {df_t.shape[1]:>6}')

    conn.close()
    return tablas

listar_tablas(DB_PATH)


  Tablas en: tabla_1.db

  TABLA                               CAPA               FILAS   COLS
  ----------------------------------- --------------- -------- ------
  data                                otra             151,445     11
  extracted_adilshamim8               extracted_       151,445     15



  Tablas en: tabla_1.db

  TABLA                               CAPA               FILAS   COLS
  ----------------------------------- --------------- -------- ------
  data                                otra             151,445     11
  extracted_adilshamim8               extracted_       151,445     15



  Tablas en: tabla_1.db

  TABLA                               CAPA               FILAS   COLS
  ----------------------------------- --------------- -------- ------
  data                                otra             151,445     11
  extracted_adilshamim8               extracted_       151,445     15



  Tablas en: tabla_1.db

  TABLA                               CAPA               FILAS   COLS
  ----------------------------------- --------------- -------- ------
  data                                otra             151,445     11
  extracted_adilshamim8               extracted_       151,445     15


['data', 'extracted_adilshamim8']

In [3]:
def inspeccionar_dataset(df, nombre='Dataset'):

    """

    Muestra un perfil básico de un DataFrame:
    dimensiones, tipos de datos, valores nulos y primeras filas.

    Parámetros:

        df     : DataFrame a inspeccionar
        nombre : etiqueta descriptiva para el encabezado

    """


    print(f'\n{"=" * 55}')
    print(f'  DATASET: {nombre}')
    print(f'{"=" * 55}')
    print(f'  Filas    : {df.shape[0]:,}')
    print(f'  Columnas : {df.shape[1]}')
    print(f'\n  Tipos de datos y valores nulos:')

    for col, dtype in df.dtypes.items():
        nulos = df[col].isna().sum()
        pct = nulos / len(df) * 100
        print(f'    {col:<28} {str(dtype):<12} nulos: {nulos} ({pct:.1f}%)')

    print(f'\n  Primeras 3 filas:')
    try:
        from IPython.display import display
        display(df.head(3))
    except ImportError:
        print(df.head(3).to_string())
    print()
# ── Leer raw_df1 en modo solo lectura ──────────────────────────────────────
conn = sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True)

df_raw = pd.read_sql_query(f'SELECT * FROM {TABLA_RAW}', conn)
conn.close()

inspeccionar_dataset(df_raw, nombre='data')


  DATASET: data
  Filas    : 151,445
  Columnas : 11

  Tipos de datos y valores nulos:
    work_year                    int64        nulos: 0 (0.0%)
    experience_level             object       nulos: 0 (0.0%)
    employment_type              object       nulos: 0 (0.0%)
    job_title                    object       nulos: 0 (0.0%)
    salary                       int64        nulos: 0 (0.0%)
    salary_currency              object       nulos: 0 (0.0%)
    salary_in_usd                int64        nulos: 0 (0.0%)
    employee_residence           object       nulos: 0 (0.0%)
    remote_ratio                 int64        nulos: 0 (0.0%)
    company_location             object       nulos: 0 (0.0%)
    company_size                 object       nulos: 0 (0.0%)

  Primeras 3 filas:


,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size
0,2025,EX,FT,Head of Data,348516,USD,348516,US,0,US,M
1,2025,EX,FT,Head of Data,232344,USD,232344,US,0,US,M
2,2025,SE,FT,Data Scientist,145400,USD,145400,US,0,US,M


In [4]:
def calcular_huella(df):

    '''Regresa la huella
    (16 caracteres) de cada
    fila del DataFrame.'''

    texto_fila = df.map(str).agg('|'.join, axis=1)              # 'v1|v2|v3|...'
    return texto_fila.map(lambda t: hashlib.sha256(t.encode('utf-8')).hexdigest()[:16])

In [5]:
prueba = df_raw.head(3).copy()
prueba['_row_hash'] = calcular_huella(df_raw.head(3))
prueba

,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size,_row_hash
0,2025,EX,FT,Head of Data,348516,USD,348516,US,0,US,M,95b566db6a71c35f
1,2025,EX,FT,Head of Data,232344,USD,232344,US,0,US,M,d3a6b1cd671b1e99
2,2025,SE,FT,Data Scientist,145400,USD,145400,US,0,US,M,7150c9650f025496


In [6]:
def extraer(etiqueta):

    # --- Datos de este dataset ---
    cfg             = tools.DATASETS[etiqueta]
    db_path         = str(cfg['db'])
    tabla_raw       = cfg['raw']
    tabla_extracted = f'extracted_{etiqueta}'
    cdc_path        = tools.RUTA_EXTRACCION_LOGS / f'cdc_{etiqueta}.json'

    try:
        # 1. Leer los datos originales (solo lectura, no se modifican)
        conn = sqlite3.connect(f'file:{db_path}?mode=ro', uri=True)
        df = pd.read_sql_query(f'SELECT * FROM {tabla_raw}', conn)

        # 2. Ver si ya existe la tabla extracted y qué huellas tiene
        existe = conn.execute(
            "SELECT COUNT(*) FROM sqlite_master WHERE type='table' AND name=?",
            (tabla_extracted,)
        ).fetchone()[0] > 0

        if existe:
            previas = set(pd.read_sql_query(f'SELECT _row_hash FROM {tabla_extracted}', conn)['_row_hash'])
        else:
            previas = set()
        conn.close()

        # 3. Calcular huellas y quedarse solo con las filas que NO se habían copiado
        df['_row_hash'] = calcular_huella(df)
        nuevas = df[~df['_row_hash'].isin(previas)].copy()
        tipo = 'full_load' if not previas else 'incremental'

        # 4. Leer la libreta de estado (FUNCIÓN 1): si no existe, se crea
        cdc = tools.logs.crear_cdc(cdc_path, etiqueta)
        batch_id = cdc.get('ultimo_batch', 0)

        # 5. Si hay filas nuevas: sumar 1 al lote, agregar las 4 columnas y copiarlas
        if len(nuevas) > 0:
            batch_id += 1
            nuevas['_batch_id']     = batch_id
            nuevas['_extracted_at'] = datetime.now().isoformat(timespec='seconds')
            nuevas['_source_table'] = tabla_raw

            columnas = [c for c in nuevas.columns if not c.startswith('_')]
            nuevas = nuevas[columnas + ['_batch_id', '_extracted_at', '_source_table', '_row_hash']]

            conn = sqlite3.connect(db_path)
            nuevas.to_sql(tabla_extracted, conn, if_exists='append', index=False)
            conn.commit()
            conn.close()

        # 6. Contar cuántas filas hay ahora en extracted
        conn = sqlite3.connect(f'file:{db_path}?mode=ro', uri=True)
        total = conn.execute(f'SELECT COUNT(*) FROM {tabla_extracted}').fetchone()[0]
        conn.close()

        # 7. Actualizar la libreta de estado (FUNCIÓN 1)
        tools.logs.crear_cdc(
            cdc_path, etiqueta,
            base_datos=cfg['db'].name,
            tabla_origen=tabla_raw,
            tabla_destino=tabla_extracted,
            ultimo_batch=batch_id,
            ultima_extraccion=datetime.now().isoformat(timespec='seconds'),
            filas_leidas=len(df),
            filas_nuevas=len(nuevas),
            filas_acumuladas=total,
            tipo=tipo,
        )

        # 8. Escribir en el diario (FUNCIÓN 2)
        tools.logs.escribir_log(
            tools.LOG_EXTRACCION,
            etiqueta=etiqueta, tabla_origen=tabla_raw, tabla_destino=tabla_extracted,
            batch_id=batch_id, filas_leidas=len(df), filas_nuevas=len(nuevas),
            filas_acumuladas=total, tipo=tipo, estado='exitoso', error=None,
        )

        print(f'[{etiqueta}] {tipo}: {len(nuevas):,} filas nuevas | total en extracted: {total:,}')

    except Exception as e:
        tools.logs.escribir_log(
            tools.LOG_EXTRACCION,
            etiqueta=etiqueta, tabla_origen=tabla_raw, tabla_destino=tabla_extracted,
            estado='error', error=str(e),
        )
        print(f'[{etiqueta}] ERROR: {e}')
        raise

## Primera ejecución: carga completa (full_load)
Como `extracted_<etiqueta>` todavía no existe, se copian todas las filas de RAW en el lote 1.

In [7]:
for etiqueta in tools.ETIQUETAS:
    extraer(etiqueta)

[adilshamim8] full_load: 151,445 filas nuevas | total en extracted: 151,445


[samithsachidanandan] full_load: 88,584 filas nuevas | total en extracted: 88,584


[saurabhbadole] full_load: 93,597 filas nuevas | total en extracted: 93,597


## Segunda ejecución: carga incremental
Se vuelve a correr la extracción. Como todas las huellas (`_row_hash`) de RAW ya existen en EXTRACTED, no se agregan filas (0 nuevas) y no se crea un nuevo lote. Esta es la evidencia del control de cambios.

In [8]:
for etiqueta in tools.ETIQUETAS:
    extraer(etiqueta)

[adilshamim8] incremental: 0 filas nuevas | total en extracted: 151,445


[samithsachidanandan] incremental: 0 filas nuevas | total en extracted: 88,584


[saurabhbadole] incremental: 0 filas nuevas | total en extracted: 93,597


### Log de extracción (una línea JSON por operación)

In [9]:
pd.read_json(tools.LOG_EXTRACCION, lines=True)

,timestamp,etiqueta,tabla_origen,tabla_destino,batch_id,filas_leidas,filas_nuevas,filas_acumuladas,tipo,estado,error
0,2026-11-07 22:43:08,adilshamim8,data,extracted_adilshamim8,1,151445,151445,151445,full_load,exitoso,NaN
1,2026-11-07 22:43:09,samithsachidanandan,data,extracted_samithsachidanandan,1,88584,88584,88584,full_load,exitoso,NaN
2,2026-11-07 22:43:09,saurabhbadole,data,extracted_saurabhbadole,1,93597,93597,93597,full_load,exitoso,NaN
3,2026-11-07 22:43:10,adilshamim8,data,extracted_adilshamim8,1,151445,0,151445,incremental,exitoso,NaN
4,2026-11-07 22:43:10,samithsachidanandan,data,extracted_samithsachidanandan,1,88584,0,88584,incremental,exitoso,NaN
5,2026-11-07 22:43:11,saurabhbadole,data,extracted_saurabhbadole,1,93597,0,93597,incremental,exitoso,NaN


In [10]:
for etiqueta in tools.ETIQUETAS:
    cfg = tools.DATASETS[etiqueta]
    conn = sqlite3.connect(f"file:{cfg['db']}?mode=ro", uri=True)
    print(f'--- extracted_{etiqueta}  ({cfg["db"].name})')
    display(pd.read_sql_query(f'SELECT * FROM extracted_{etiqueta} LIMIT 3', conn))
    conn.close()

--- extracted_adilshamim8  (tabla_1.db)


,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size,_batch_id,_extracted_at,_source_table,_row_hash
0,2025,EX,FT,Head of Data,348516,USD,348516,US,0,US,M,1,2026-11-07T22:43:08,data,95b566db6a71c35f
1,2025,EX,FT,Head of Data,232344,USD,232344,US,0,US,M,1,2026-11-07T22:43:08,data,d3a6b1cd671b1e99
2,2025,SE,FT,Data Scientist,145400,USD,145400,US,0,US,M,1,2026-11-07T22:43:08,data,7150c9650f025496


--- extracted_samithsachidanandan  (tabla_2.db)


,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size,_batch_id,_extracted_at,_source_table,_row_hash
0,2025,MI,FT,Customer Success Manager,57000,EUR,60000,NL,50,NL,L,1,2026-11-07T22:43:09,data,261674dcbcdcf1a4
1,2025,SE,FT,Engineer,165000,USD,165000,US,0,US,M,1,2026-11-07T22:43:09,data,a3b0ad9afbcbbed8
2,2025,SE,FT,Engineer,109000,USD,109000,US,0,US,M,1,2026-11-07T22:43:09,data,55455f9cefa2ba22


--- extracted_saurabhbadole  (tabla_3.db)


,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size,_batch_id,_extracted_at,_source_table,_row_hash
0,2025,MI,FT,Research Scientist,208000,USD,208000,US,0,US,M,1,2026-11-07T22:43:09,data,d9257db080df5019
1,2025,MI,FT,Research Scientist,147000,USD,147000,US,0,US,M,1,2026-11-07T22:43:09,data,6de8ed3c6ff836fc
2,2025,SE,FT,Research Scientist,173000,USD,173000,US,0,US,M,1,2026-11-07T22:43:09,data,a0ae077a49dbd4af


In [11]:
for etiqueta in tools.ETIQUETAS:
    ruta = tools.RUTA_EXTRACCION_LOGS / f'cdc_{etiqueta}.json'
    print(f'--- cdc_{etiqueta}.json')
    print(ruta.read_text(encoding='utf-8'))

--- cdc_adilshamim8.json
{
  "etiqueta": "adilshamim8",
  "actualizado_en": "2026-11-07T22:43:10",
  "base_datos": "tabla_1.db",
  "tabla_origen": "data",
  "tabla_destino": "extracted_adilshamim8",
  "ultimo_batch": 1,
  "ultima_extraccion": "2026-11-07T22:43:10",
  "filas_leidas": 151445,
  "filas_nuevas": 0,
  "filas_acumuladas": 151445,
  "tipo": "incremental"
}
--- cdc_samithsachidanandan.json
{
  "etiqueta": "samithsachidanandan",
  "actualizado_en": "2026-11-07T22:43:10",
  "base_datos": "tabla_2.db",
  "tabla_origen": "data",
  "tabla_destino": "extracted_samithsachidanandan",
  "ultimo_batch": 1,
  "ultima_extraccion": "2026-11-07T22:43:10",
  "filas_leidas": 88584,
  "filas_nuevas": 0,
  "filas_acumuladas": 88584,
  "tipo": "incremental"
}
--- cdc_saurabhbadole.json
{
  "etiqueta": "saurabhbadole",
  "actualizado_en": "2026-11-07T22:43:11",
  "base_datos": "tabla_3.db",
  "tabla_origen": "data",
  "tabla_destino": "extracted_saurabhbadole",
  "ultimo_batch": 1,
  "ultima_extr

In [12]:
for etiqueta in tools.ETIQUETAS:
    cfg = tools.DATASETS[etiqueta]
    conn = sqlite3.connect(f"file:{cfg['db']}?mode=ro", uri=True)
    print(f'\n===== {etiqueta}  ({cfg["db"].name}) =====')

    print('1) RAW vs EXTRACTED:')
    display(pd.read_sql_query(f'''
        SELECT (SELECT COUNT(*) FROM {cfg['raw']})                           AS filas_raw,
               (SELECT COUNT(*) FROM extracted_{etiqueta})                   AS filas_extracted,
               (SELECT COUNT(DISTINCT _row_hash) FROM extracted_{etiqueta})  AS huellas_distintas
    ''', conn))

    print('2) Filas por lote:')
    display(pd.read_sql_query(f'''
        SELECT _batch_id, _extracted_at, _source_table, COUNT(*) AS filas
        FROM extracted_{etiqueta}
        GROUP BY _batch_id, _extracted_at, _source_table
        ORDER BY _batch_id
    ''', conn))

    print('3) Metadatos sin nulos:')
    display(pd.read_sql_query(f'''
        SELECT COUNT(*) AS filas,
               SUM(CASE WHEN _batch_id     IS NULL THEN 1 ELSE 0 END) AS nulos_batch,
               SUM(CASE WHEN _extracted_at IS NULL THEN 1 ELSE 0 END) AS nulos_fecha,
               SUM(CASE WHEN _source_table IS NULL THEN 1 ELSE 0 END) AS nulos_origen,
               SUM(CASE WHEN _row_hash     IS NULL THEN 1 ELSE 0 END) AS nulos_huella
        FROM extracted_{etiqueta}
    ''', conn))

    conn.close()


===== adilshamim8  (tabla_1.db) =====
1) RAW vs EXTRACTED:


,filas_raw,filas_extracted,huellas_distintas
0,151445,151445,71913


2) Filas por lote:


,_batch_id,_extracted_at,_source_table,filas
0,1,2026-11-07T22:43:08,data,151445


3) Metadatos sin nulos:


,filas,nulos_batch,nulos_fecha,nulos_origen,nulos_huella
0,151445,0,0,0,0


... (8 outputs left out)

2) Filas por lote:


,_batch_id,_extracted_at,_source_table,filas
0,1,2026-11-07T22:43:09,data,93597


3) Metadatos sin nulos:


,filas,nulos_batch,nulos_fecha,nulos_origen,nulos_huella
0,93597,0,0,0,0
